In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import lightning as L
from torch.utils.data import TensorDataset, DataLoader

In [2]:
class LSTMbyHand(L.LightningModule):
    
    def __init__(self):

        super().__init__()

        mean = torch.tensor(0.0)
        std = torch.tensor(1.0)

        self.wlr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wlr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.blr1 = nn.Parameter(torch.tensor(0.0), requires_grad=True) 

        self.wpr1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wpr2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bpr1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

        self.wpl1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wpl2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bpl1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

        self.wpm1 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.wpm2 = nn.Parameter(torch.normal(mean=mean, std=std), requires_grad=True)
        self.bpm1 = nn.Parameter(torch.tensor(0.0), requires_grad=True)

    def lstm_unit(self, input_value, long_memory, short_memory):

        long_remember_percent = torch.sigmoid((short_memory*self.wlr1)+(input_value*self.wlr2) + self.blr1)
        potential_remember_percent = torch.sigmoid((short_memory*self.wpr1)+(input_value*self.wpr2)+self.bpr1)
        potential_memory = torch.tanh((short_memory*self.wpl1)+(input_value*self.wpl2) + self.bpl1)
        
        updated_long_memory = (long_memory*long_remember_percent) + (potential_memory*potential_remember_percent)
        output_percent = torch.sigmoid((short_memory*self.wpm1)+(input_value*self.wpm2)+self.bpm1)
        
        updated_short_memory = torch.tanh(updated_long_memory) * output_percent

        return ([updated_long_memory, updated_short_memory])
    
    def forward(self, input):

        long_memory = 0
        short_memory = 0
        day1 = input[1]
        day2 = input[2]
        day3 = input[3]
        day4 = input[4]

        long_memory, short_memory = self.lstm_unit(day1, long_memory, short_memory)
        long_memory, short_memory = self.lstm_unit(day2, long_memory, short_memory)
        long_memory, short_memory = self.lstm_unit(day3, long_memory, short_memory)
        long_memory, short_memory = self.lstm_unit(day4, long_memory, short_memory)

        return short_memory
    
    def configure_optimizers(self):
        return Adam(self.parameters())
    
    def training_step(self, batch, batch_idx):
        
        input_i, label_i = batch
        output_i = self.forward(input_i[0])
        loss = (output_i-input_i)**2

        self.log("train_loss", loss)

        if label_i == 0:
            self.log("out_0", output_i)
        else:
            self.log("out_1", output_i)

In [3]:
model = LSTMbyHand()

In [ ]:
print("\nNow Let's compare the observed and predicted value...\n")
print("Company A: Observed = 0, Predicted = ", model(torch.tensor([0.0, 0.50, 0.25, 1.0])).detach())
print("Company B: Observed = 0, Predicted = ", model(torch.tensor([0.0, 0.25, 0.50, 1.0])).detach())